# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [3]:
context = [{'role':'system', 'content':"""
You are an expert SQL assistant.
Generate only valid SQLite SQL queries.

Database schema:

CREATE TABLE employees (
    ID_usr INTEGER PRIMARY KEY,
    name TEXT
);

CREATE TABLE salary (
    ID_usr INTEGER,
    year DATE,
    salary REAL,
    FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);

CREATE TABLE studies (
    ID INTEGER PRIMARY KEY,
    ID_usr INTEGER,
    educational_level INTEGER,
    Institution TEXT,
    Years DATE,
    Speciality TEXT,
    FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);

Sample rows:

employees:
ID_usr | name
1      | Ana
2      | John
3      | Maria

salary:
ID_usr | year       | salary
1      | 2022-01-01 | 45000
1      | 2023-01-01 | 50000
2      | 2023-01-01 | 60000

studies:
ID | ID_usr | educational_level | Institution | Years      | Speciality
1  | 1      | 3                 | Ironhack    | 2020-01-01 | Data Analytics
2  | 2      | 4                 | MIT         | 2019-01-01 | Computer Science
3  | 3      | 2                 | UCM         | 2021-01-01 | Business

Rules:
- Use only the tables and columns listed above.
- Use ID_usr to join employees with salary and studies.
- Do not invent columns or tables.
- Return only the SQL query.
""" }]

In [4]:
# FEW SHOT SAMPLES
context.append({'role':'system', 'content':"""
Maintain the SQL query simple and efficient.
Use valid SQLite.
Return only the SQL query.

Examples:

Question: Show all employee names.
SQL:
SELECT name
FROM employees;

Question: Show each employee name with their salary.
SQL:
SELECT e.name, s.salary
FROM employees e
JOIN salary s
    ON e.ID_usr = s.ID_usr;

Question: Show employees who studied at Ironhack.
SQL:
SELECT e.name, st.Institution, st.Speciality
FROM employees e
JOIN studies st
    ON e.ID_usr = st.ID_usr
WHERE st.Institution = 'Ironhack';

Question: Show the average salary by year.
SQL:
SELECT year, AVG(salary) AS average_salary
FROM salary
GROUP BY year;

Question: Show each employee's highest salary.
SQL:
SELECT e.name, MAX(s.salary) AS highest_salary
FROM employees e
JOIN salary s
    ON e.ID_usr = s.ID_usr
GROUP BY e.ID_usr, e.name;
"""})

In [5]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [8]:
#new
context_user = context.copy()
print(return_CCRMSQL("Show the names of employees and their salaries in 2023.", context_user))

SQL:
SELECT e.name, s.salary
FROM employees e
JOIN salary s
    ON e.ID_usr = s.ID_usr
WHERE s.year = '2023-01-01';


In [9]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Show the names of employees and their salaries in 2023.", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2023';
```

This SQL query selects the names of employees and their salaries from the "employees" and "salary" tables, joining them on the ID_usr field. It filters the results to only show salaries from the year 2023.


In [10]:
#new
print(return_CCRMSQL("Which employees studied Computer Science?", context_user))

SQL:
SELECT e.name
FROM employees e
JOIN studies st
    ON e.ID_usr = st.ID_usr
WHERE st.Speciality = 'Computer Science';


In [11]:
#old
print(return_CCRMSQL("Which employees studied Computer Science?", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
WHERE s.Speciality = 'Computer Science';
```

This SQL query selects the names of employees who studied Computer Science by joining the "employees" table with the "studies" table on the ID_usr field and filtering the results based on the Speciality column being 'Computer Science'.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [12]:
#new
print(return_CCRMSQL("Show the average salary for each year.", context_user))

SQL:
SELECT year, AVG(salary) AS average_salary
FROM salary
GROUP BY year;


In [13]:
#old
print(return_CCRMSQL("Show the average salary for each year.", old_context_user))

This is your SQL:
```sql
SELECT year, AVG(salary) AS average_salary
FROM salary
GROUP BY year;
```

This SQL query selects the year and calculates the average salary for each year from the "salary" table. The results are grouped by year using the GROUP BY clause, and the AVG() function is used to calculate the average salary for each group.


In [14]:
#new
print(return_CCRMSQL("Show employees with salaries higher than 50000.", context_user))

SQL:
SELECT e.name
FROM employees e
JOIN salary s
    ON e.ID_usr = s.ID_usr
WHERE s.salary > 50000;


In [15]:
#old
print(return_CCRMSQL("Show employees with salaries higher than 50000.", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary > 50000;
```

This SQL query selects the names of employees from the "employees" table who have salaries higher than 50000 from the "salary" table by joining the two tables on the ID_usr column.


In [16]:
#new
print(return_CCRMSQL("Show employee names and the institutions where they studied.", context_user))

SQL:
SELECT e.name, st.Institution
FROM employees e
JOIN studies st
    ON e.ID_usr = st.ID_usr;


In [17]:
#old
print(return_CCRMSQL("Show employee names and the institutions where they studied.", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.Institution
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr;
```

This SQL query selects the employee names and the institutions where they studied by joining the "employees" table with the "studies" table on the common column "ID_usr".


# Report: Prompt Engineering for Text-to-SQL Generation

In this exercise, I compared different prompt engineering approaches for generating SQL queries from natural language questions using GPT-3.5-Turbo. The goal was to evaluate how prompt structure affects the quality, accuracy, and consistency of the generated SQL queries.

The first approach used an older prompt format where the database schema was described using custom JSON-like text structures. The prompt mainly focused on instructions such as asking the model to keep the SQL simple and explain the result afterward. Although this approach worked for some simple queries, it had several limitations. The schema descriptions were inconsistent because some fields used different names such as `nombre` instead of `name`, and some tables had formatting issues. This made the schema less clear for the model and increased the chance of mistakes or hallucinations.

The second approach used a newer prompt structure inspired by the paper *How to Prompt LLMs for Text-to-SQL*. In this version, the schema was written using actual SQL `CREATE TABLE` statements, sample rows were added, and few-shot examples were included. This created a more realistic and structured environment for the model. The prompt also explicitly instructed the model to return only valid SQLite SQL.

The results showed that the new prompt consistently produced cleaner and more reliable SQL queries. For example, when asking for salaries in 2023, the new prompt correctly generated:

```sql
WHERE s.year = '2023-01-01'
```

because the sample rows showed that dates were stored as full date strings. The old prompt generated:

```sql
WHERE s.year = '2023'
```

which may fail depending on the database format. This demonstrated that sample rows help the model better understand the actual data representation.

For simpler queries, such as filtering salaries above 50000 or calculating average salaries, both prompts often produced correct SQL queries. However, the old prompt always included extra explanations and markdown formatting, while the new prompt returned cleaner SQL that would be easier to execute automatically in a real application.

One important observation was that not all improvements were related to correctness. In several cases, both prompts produced valid SQL, but the new prompt generated more readable formatting, clearer aliases, and more standardized SQL style. This improved consistency across responses.

Some variations did not work well. The old prompt occasionally encouraged unnecessary text generation because it requested explanations and formatting instructions. In addition, the inconsistent schema definitions sometimes confused the model and increased the risk of hallucinated columns or incorrect assumptions. Another weakness was that the old prompt did not provide examples of expected SQL outputs, so the model had less guidance on query structure.

The main lesson I learned is that large language models perform much better when prompts are highly structured and closely resemble real-world syntax. Using SQL CREATE TABLE statements, realistic sample rows, and few-shot examples significantly improves accuracy and reduces hallucinations. I also learned that prompt engineering is not only about correctness, but also about improving consistency, readability, and usability of the generated results. Structured prompts provide stronger context and help the model generate outputs that are more suitable for real Text-to-SQL systems.
